# Step 01 — Ingest & Structural Profile

**Berka / PKDD'99 Financial Dataset → DuckDB**

Loads the 8 raw `.asc` files with every column as text, then audits the file before a single
type cast is made. Nothing is cleaned here. The only output is a database of raw tables plus a
profile that tells Step 02 what it is allowed to assume.

Exit criteria for this step:

- Row count per table matches the published figure
- No duplicate primary keys, no orphan foreign keys
- Missingness mapped per column, and blank-vs-NULL distinguished
- Every categorical code enumerated
- Every date column parses with a known format, ranges plausible
- Numeric ranges checked for impossible values


## 1.0 Setup


In [2]:
!pip install -q duckdb


In [3]:
import duckdb, pandas as pd, shutil, os
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/berka-banking-analytics'
RAW  = f'{BASE}/Data/raw'
OUT  = f'{BASE}/Step01_Ingest'
DB   = '/content/berka.duckdb'

os.makedirs(OUT, exist_ok=True)
pd.set_option('display.max_rows', 100)
print(duckdb.__version__, sorted(os.listdir(RAW)))


Mounted at /content/drive
1.3.2 ['account.asc', 'card.asc', 'client.asc', 'disp.asc', 'district.asc', 'loan.asc', 'order.asc', 'trans.asc']


DuckDB writes to local disk, not the Drive mount. The FUSE mount does not support the file
locking DuckDB expects and will fail intermittently on a 1M-row insert. The finished `.duckdb`
is copied to Drive at the end of the step.


## 1.1 Ingest

`all_varchar=true` is deliberate. Loading as text means the CSV reader cannot silently coerce a
malformed value to NULL, so every cast becomes an explicit decision in Step 02 with a visible
failure. `date` columns in particular would be read as integers and lose the `YYMMDD` structure.


In [4]:
TABLES = {'account':4500, 'card':892, 'client':5369, 'disp':5369,
          'district':77, 'loan':682, 'order':6471, 'trans':1056320}

con = duckdb.connect(DB)
for t in TABLES:
    con.execute(f"""CREATE OR REPLACE TABLE raw_{t} AS
                    SELECT * FROM read_csv('{RAW}/{t}.asc',
                                           delim=';', header=true, all_varchar=true)""")
print('loaded')


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

loaded


## 1.2 Row counts against the published figures


In [5]:
counts = pd.DataFrame(
    [(t, con.sql(f'SELECT count(*) FROM raw_{t}').fetchone()[0], n) for t, n in TABLES.items()],
    columns=['table', 'loaded', 'expected'])
counts['match'] = counts.loaded == counts.expected
counts


,table,loaded,expected,match
0,account,4500,4500,True
1,card,892,892,True
2,client,5369,5369,True
3,disp,5369,5369,True
4,district,77,77,True
5,loan,682,682,True
6,order,6471,6471,True
7,trans,1056320,1056320,True


## 1.3 Keys

Duplicate primary keys and orphan foreign keys both break the star schema silently: a duplicate
key inflates a fact join, an orphan drops rows on an inner join. Both are checked before design,
not after.


In [6]:
con.sql("""
SELECT 'account'  t, count(*) - count(DISTINCT account_id) dup_keys FROM raw_account
UNION ALL SELECT 'client', count(*) - count(DISTINCT client_id)    FROM raw_client
UNION ALL SELECT 'disp',   count(*) - count(DISTINCT disp_id)      FROM raw_disp
UNION ALL SELECT 'card',   count(*) - count(DISTINCT card_id)      FROM raw_card
UNION ALL SELECT 'loan',   count(*) - count(DISTINCT loan_id)      FROM raw_loan
UNION ALL SELECT 'order',  count(*) - count(DISTINCT order_id)     FROM raw_order
UNION ALL SELECT 'trans',  count(*) - count(DISTINCT trans_id)     FROM raw_trans
UNION ALL SELECT 'district', count(*) - count(DISTINCT A1)         FROM raw_district
""").df()


,t,dup_keys
0,account,0
1,client,0
2,disp,0
3,card,0
4,loan,0
5,order,0
6,trans,0
7,district,0


In [7]:
con.sql("""
SELECT 'trans.account_id -> account' rel, count(*) orphans
  FROM raw_trans t LEFT JOIN raw_account a USING(account_id) WHERE a.account_id IS NULL
UNION ALL SELECT 'loan.account_id -> account', count(*)
  FROM raw_loan l LEFT JOIN raw_account a USING(account_id) WHERE a.account_id IS NULL
UNION ALL SELECT 'order.account_id -> account', count(*)
  FROM raw_order o LEFT JOIN raw_account a USING(account_id) WHERE a.account_id IS NULL
UNION ALL SELECT 'disp.client_id -> client', count(*)
  FROM raw_disp d LEFT JOIN raw_client c USING(client_id) WHERE c.client_id IS NULL
UNION ALL SELECT 'disp.account_id -> account', count(*)
  FROM raw_disp d LEFT JOIN raw_account a USING(account_id) WHERE a.account_id IS NULL
UNION ALL SELECT 'card.disp_id -> disp', count(*)
  FROM raw_card k LEFT JOIN raw_disp d USING(disp_id) WHERE d.disp_id IS NULL
UNION ALL SELECT 'account.district_id -> district', count(*)
  FROM raw_account a LEFT JOIN raw_district d ON a.district_id = d.A1 WHERE d.A1 IS NULL
UNION ALL SELECT 'client.district_id -> district', count(*)
  FROM raw_client c LEFT JOIN raw_district d ON c.district_id = d.A1 WHERE d.A1 IS NULL
""").df()


,rel,orphans
0,trans.account_id -> account,0
1,loan.account_id -> account,0
2,order.account_id -> account,0
3,disp.client_id -> client,0
4,disp.account_id -> account,0
5,card.disp_id -> disp,0
6,account.district_id -> district,0
7,client.district_id -> district,0


## 1.4 Missingness

Three different things read as "missing" in these files and they are not interchangeable: a true
`NULL`, an empty-after-trim string, and the literal `?` used in `district`. Counting them
separately now prevents a category that is 50% unknown from looking populated in Power BI.


In [8]:
rows = []
for t in TABLES:
    cols = [c[0] for c in con.sql(f'DESCRIBE raw_{t}').fetchall()]
    n = con.sql(f'SELECT count(*) FROM raw_{t}').fetchone()[0]
    for c in cols:
        nn, nb, nq = con.sql(f"""SELECT count(*) FILTER (WHERE "{c}" IS NULL),
                                        count(*) FILTER (WHERE trim("{c}") = ''),
                                        count(*) FILTER (WHERE trim("{c}") = '?')
                                 FROM raw_{t}""").fetchone()
        if nn or nb or nq:
            rows.append((t, c, n, nn, nb, nq, round(100*(nn+nb+nq)/n, 1)))

missing = pd.DataFrame(rows, columns=['table','column','rows','null','blank','question','pct_missing'])
missing.sort_values('pct_missing', ascending=False)


,table,column,rows,null,blank,question,pct_missing
5,trans,bank,1056320,782812,0,0,74.1
6,trans,account,1056320,760931,0,0,72.0
4,trans,k_symbol,1056320,481881,53433,0,50.7
2,order,k_symbol,6471,0,1379,0,21.3
3,trans,operation,1056320,183114,0,0,17.3
1,district,A15,77,0,0,1,1.3
0,district,A12,77,0,0,1,1.3


## 1.5 Categorical code inventory

Every code is Czech and every one becomes a dimension attribute in Step 02. Enumerating them here
means the translation map is built from what is in the file, not from the data dictionary, which
is the version that matters.


In [9]:
for t, c in [('trans','type'), ('trans','operation'), ('trans','k_symbol'),
             ('order','k_symbol'), ('account','frequency'),
             ('disp','type'), ('card','type'), ('loan','status')]:
    display(con.sql(f"""SELECT '{t}.{c}' col,
                               coalesce(nullif(trim("{c}"), ''), '<MISSING>') code,
                               count(*) n,
                               round(100.0*count(*)/sum(count(*)) OVER (), 1) pct
                        FROM raw_{t} GROUP BY 2 ORDER BY n DESC""").df())


,col,code,n,pct
0,trans.type,VYDAJ,634571,60.1
1,trans.type,PRIJEM,405083,38.3
2,trans.type,VYBER,16666,1.6


,col,code,n,pct
0,trans.operation,VYBER,434918,41.2
1,trans.operation,PREVOD NA UCET,208283,19.7
2,trans.operation,<MISSING>,183114,17.3
3,trans.operation,VKLAD,156743,14.8
4,trans.operation,PREVOD Z UCTU,65226,6.2
5,trans.operation,VYBER KARTOU,8036,0.8


,col,code,n,pct
0,trans.k_symbol,<MISSING>,535314,50.7
1,trans.k_symbol,UROK,183114,17.3
2,trans.k_symbol,SLUZBY,155832,14.8
3,trans.k_symbol,SIPO,118065,11.2
4,trans.k_symbol,DUCHOD,30338,2.9
5,trans.k_symbol,POJISTNE,18500,1.8
6,trans.k_symbol,UVER,13580,1.3
7,trans.k_symbol,SANKC. UROK,1577,0.1


,col,code,n,pct
0,order.k_symbol,SIPO,3502,54.1
1,order.k_symbol,<MISSING>,1379,21.3
2,order.k_symbol,UVER,717,11.1
3,order.k_symbol,POJISTNE,532,8.2
4,order.k_symbol,LEASING,341,5.3


,col,code,n,pct
0,account.frequency,POPLATEK MESICNE,4167,92.6
1,account.frequency,POPLATEK TYDNE,240,5.3
2,account.frequency,POPLATEK PO OBRATU,93,2.1


,col,code,n,pct
0,disp.type,OWNER,4500,83.8
1,disp.type,DISPONENT,869,16.2


,col,code,n,pct
0,card.type,classic,659,73.9
1,card.type,junior,145,16.3
2,card.type,gold,88,9.9


,col,code,n,pct
0,loan.status,C,403,59.1
1,loan.status,A,203,29.8
2,loan.status,D,45,6.6
3,loan.status,B,31,4.5


`trans.type` is worth a second look. It should be a two-value field (credit / debit) and it is not.


In [10]:
con.sql("""
SELECT type, coalesce(operation, '<NULL>') operation, count(*) n
FROM raw_trans GROUP BY 1, 2 ORDER BY 1, 3 DESC
""").df()


,type,operation,n
0,PRIJEM,<NULL>,183114
1,PRIJEM,VKLAD,156743
2,PRIJEM,PREVOD Z UCTU,65226
3,VYBER,VYBER,16666
4,VYDAJ,VYBER,418252
5,VYDAJ,PREVOD NA UCET,208283
6,VYDAJ,VYBER KARTOU,8036


In [11]:
con.sql("""
SELECT count(*) FILTER (WHERE operation IS NULL)                              op_null,
       count(*) FILTER (WHERE operation IS NULL AND trim(k_symbol) = 'UROK')  op_null_and_urok
FROM raw_trans
""").df()


,op_null,op_null_and_urok
0,183114,183114


## 1.6 Dates

Every date is `YYMMDD` and every year falls in 1911–1998, so `19` can be prefixed without
ambiguity. `card.issued` carries a time suffix the other columns do not, and parsing it with the
bare `%y%m%d` format returns NULL rather than raising, which is exactly the kind of silent loss
`all_varchar` was meant to expose.


In [12]:
con.sql("""
SELECT 'account.date' col, count(*) n, count(*) FILTER (WHERE try_strptime(date, '%y%m%d') IS NULL) unparsed,
       min(strptime(date, '%y%m%d'))::DATE lo, max(strptime(date, '%y%m%d'))::DATE hi FROM raw_account
UNION ALL SELECT 'trans.date', count(*), count(*) FILTER (WHERE try_strptime(date, '%y%m%d') IS NULL),
       min(strptime(date, '%y%m%d'))::DATE, max(strptime(date, '%y%m%d'))::DATE FROM raw_trans
UNION ALL SELECT 'loan.date', count(*), count(*) FILTER (WHERE try_strptime(date, '%y%m%d') IS NULL),
       min(strptime(date, '%y%m%d'))::DATE, max(strptime(date, '%y%m%d'))::DATE FROM raw_loan
UNION ALL SELECT 'card.issued', count(*), count(*) FILTER (WHERE try_strptime(issued, '%y%m%d %H:%M:%S') IS NULL),
       min(strptime(issued, '%y%m%d %H:%M:%S'))::DATE, max(strptime(issued, '%y%m%d %H:%M:%S'))::DATE FROM raw_card
""").df()


,col,n,unparsed,lo,hi
0,account.date,4500,0,1993-01-01,1997-12-29
1,trans.date,1056320,0,1993-01-01,1998-12-31
2,loan.date,682,0,1993-07-05,1998-12-08
3,card.issued,892,0,1993-11-07,1998-12-29


In [13]:
con.sql("SELECT count(*) card_issued_lost_by_wrong_format FROM raw_card WHERE try_strptime(issued, '%y%m%d') IS NULL").df()


,card_issued_lost_by_wrong_format
0,892


## 1.7 `birth_number` decode

Gender is encoded by adding 50 to the month. The decode is only safe if every raw month lands in
1–12 or 51–62 and every resulting date is real, so both are checked rather than assumed.


In [14]:
con.sql("""
WITH d AS (SELECT client_id,
                  CAST(substr(birth_number,1,2) AS INT) yy,
                  CAST(substr(birth_number,3,2) AS INT) mm,
                  CAST(substr(birth_number,5,2) AS INT) dd
           FROM raw_client)
SELECT count(*) FILTER (WHERE mm NOT BETWEEN 1 AND 12 AND mm NOT BETWEEN 51 AND 62) bad_month,
       count(*) FILTER (WHERE dd NOT BETWEEN 1 AND 31)                              bad_day,
       count(*) FILTER (WHERE try_cast(concat_ws('-', 1900+yy,
                             CASE WHEN mm > 50 THEN mm-50 ELSE mm END, dd) AS DATE) IS NULL) bad_date
FROM d
""").df()


,bad_month,bad_day,bad_date
0,0,0,0


In [15]:
con.sql("""
WITH d AS (SELECT CAST(substr(birth_number,1,2) AS INT) yy,
                  CAST(substr(birth_number,3,2) AS INT) mm,
                  CAST(substr(birth_number,5,2) AS INT) dd
           FROM raw_client)
SELECT CASE WHEN mm > 50 THEN 'Female' ELSE 'Male' END gender, count(*) n,
       min(make_date(1900+yy, CASE WHEN mm > 50 THEN mm-50 ELSE mm END, dd)) earliest,
       max(make_date(1900+yy, CASE WHEN mm > 50 THEN mm-50 ELSE mm END, dd)) latest
FROM d GROUP BY 1
""").df()


,gender,n,earliest,latest
0,Female,2645,1914-03-01,1987-09-27
1,Male,2724,1911-08-20,1986-08-13


## 1.8 Numeric ranges

A negative balance is legitimate (overdraft) and stays. A non-positive transaction amount is not,
and needs a decision in Step 02 rather than a silent pass.


In [16]:
con.sql("""
SELECT round(min(CAST(amount AS DOUBLE)),2)  amt_min,  round(max(CAST(amount AS DOUBLE)),2)  amt_max,
       round(min(CAST(balance AS DOUBLE)),2) bal_min,  round(max(CAST(balance AS DOUBLE)),2) bal_max,
       count(*) FILTER (WHERE CAST(amount AS DOUBLE)  <= 0) nonpositive_amount,
       count(*) FILTER (WHERE CAST(balance AS DOUBLE) <  0) negative_balance
FROM raw_trans
""").df()


,amt_min,amt_max,bal_min,bal_max,nonpositive_amount,negative_balance
0,0.0,87400.0,-41125.7,209637.0,14,2999


**Balance reconciliation.** `balance` is a recorded running total. Before Step 03 builds a monthly
snapshot on top of it, it has to be established whether it agrees with the signed transaction
amounts, and which sign rule is correct given the three-valued `type`.


In [17]:
con.sql("""
WITH s AS (
  SELECT account_id,
         SUM(CASE WHEN type = 'PRIJEM' THEN CAST(amount AS DOUBLE)
                  ELSE -CAST(amount AS DOUBLE) END) signed_sum,
         CAST(argmax(balance, CAST(date AS BIGINT)*100000 + CAST(trans_id AS BIGINT)) AS DOUBLE) final_balance
  FROM raw_trans GROUP BY 1)
SELECT count(*) accounts,
       count(*) FILTER (WHERE abs(final_balance - signed_sum) <= 1)   within_1_czk,
       count(*) FILTER (WHERE abs(final_balance - signed_sum) >  1)   over_1_czk,
       count(*) FILTER (WHERE abs(final_balance - signed_sum) >  100) over_100_czk,
       round(median(abs(final_balance - signed_sum)), 2)              median_gap,
       round(max(abs(final_balance - signed_sum)), 2)                 max_gap
FROM s
""").df()


,accounts,within_1_czk,over_1_czk,over_100_czk,median_gap,max_gap
0,4500,4355,145,55,0.2,1200.0


## 1.9 Coverage and grain

Which entities actually carry facts, and at what grain. This determines whether `loan` is a fact
table or a degenerate attribute of the account, and whether the client–account link genuinely
needs a bridge.


In [18]:
con.sql("""
SELECT (SELECT count(*) FROM raw_account)                          accounts,
       (SELECT count(DISTINCT account_id) FROM raw_trans)          with_transactions,
       (SELECT count(DISTINCT account_id) FROM raw_order)          with_standing_order,
       (SELECT count(DISTINCT account_id) FROM raw_loan)           with_loan,
       (SELECT count(*) FROM raw_loan)                             loans,
       (SELECT count(*) FROM raw_disp WHERE type = 'OWNER')        owner_dispositions,
       (SELECT count(*) FROM raw_disp WHERE type = 'DISPONENT')    disponent_dispositions
""").df()


,accounts,with_transactions,with_standing_order,with_loan,loans,owner_dispositions,disponent_dispositions
0,4500,4500,3758,682,682,4500,869


In [19]:
con.sql("""
SELECT (SELECT count(*) FROM (SELECT account_id FROM raw_loan GROUP BY 1 HAVING count(*) > 1)) accounts_with_multiple_loans,
       (SELECT count(*) FROM (SELECT account_id FROM raw_disp WHERE type='OWNER' GROUP BY 1 HAVING count(*) > 1)) accounts_with_multiple_owners,
       (SELECT count(*) FROM (SELECT client_id  FROM raw_disp GROUP BY 1 HAVING count(*) > 1)) clients_on_multiple_accounts
""").df()


,accounts_with_multiple_loans,accounts_with_multiple_owners,clients_on_multiple_accounts
0,0,0,0


In [20]:
con.sql("""
SELECT d.type disposition_type, count(*) cards
FROM raw_card k JOIN raw_disp d USING(disp_id) GROUP BY 1
""").df()


,disposition_type,cards
0,OWNER,892


## 1.10 Persist


In [21]:
missing.to_csv(f'{OUT}/profile_missingness.csv', index=False)
counts.to_csv(f'{OUT}/profile_rowcounts.csv', index=False)
con.close()
os.makedirs(f'{BASE}/Data', exist_ok=True)
shutil.copy(DB, f'{BASE}/Data/berka.duckdb')
print('raw database →', f'{BASE}/Data/berka.duckdb')
print(os.listdir(OUT))

raw database → /content/drive/MyDrive/berka-banking-analytics/Data/berka.duckdb
['profile_missingness.csv', 'profile_rowcounts.csv', 'berka.duckdb']


---

## Findings carried into Step 02

| # | Finding | Consequence for the model |
|---|---|---|
| 1 | All 8 row counts match; 0 duplicate keys; 0 orphan foreign keys | Joins are safe, no defensive filtering needed |
| 2 | `trans.type` has three values, not two — 16,666 rows coded `VYBER` instead of `VYDAJ` | Sign rule must be `PRIJEM = credit, everything else = debit`. Testing the alternative reconciles worse, so this is settled empirically, not by the dictionary |
| 3 | `operation IS NULL` ⇔ `k_symbol = 'UROK'`, zero exceptions | Not missing data. It is interest crediting, and gets its own explicit category |
| 4 | `trans.k_symbol` unknown for ~51% of rows, split across NULL and blank | Payment-purpose analysis is restricted to the identified half, and this must be stated on the dashboard page, not hidden |
| 5 | `card.issued` carries a time suffix; the other date columns do not | Separate parse format. Using `%y%m%d` loses all 892 rows to NULL without error |
| 6 | `district` 69 has `?` for unemployment '95 and crime '95 | 1 of 77 rows. Cast with `try_cast`, keep the district, flag the two measures as unavailable for that year |
| 7 | Exactly one `OWNER` per account; 869 `DISPONENT` rows; clients appear on multiple accounts | Client↔account is genuinely many-to-many. The bridge is required, not decorative |
| 8 | 682 loans across 682 distinct accounts — no account has two | `fact_loan` is 1:1 with account. Still modelled as a fact, but loan-level and account-level measures cannot double-count |
| 9 | All 892 cards sit on `OWNER` dispositions | Card is an owner attribute; no disponent card path to handle |
| 10 | Balance reconciles to signed amounts within 1 CZK for 4,357 of 4,500 accounts; 55 exceed 100 CZK | Recorded `balance` is authoritative for the Step 03 snapshot. Do not recompute it as a running sum |
| 11 | Transactions span 1993-01-01 to 1998-12-31; accounts opened to 1997-12-29 | `dim_date` spine is 1993-01-01 → 1998-12-31. Not 1999 |
| 12 | 2,999 negative balances, 14 non-positive amounts | Overdrafts are real and kept. The 14 amounts need a decision in Step 02 |

**Next:** Step 02 builds the star schema — typed dimensions, the disposition bridge, the junk
dimension collapsing `type` × `operation` × `k_symbol`, and the date spine.
